# phase_3 (M3 C-KAN) on Kaggle — Drive-resumable

Trains the per-region concept→disease model on **precomputed BioViL-T features**
(M1 is frozen, so features are cached once and just loaded here).

**Storage:** `/kaggle/working` is wiped between sessions unless you Commit, and a
session can die. We push the run dir to **Google Drive (rclone)** each epoch (and
every `SYNC_EVERY` steps); next session set `RESUME=1` to pull `last.pt` and continue.

**Inputs you must upload as Kaggle datasets:**
1. `phase_3/` code folder (bundles the two small label JSONs — self-contained).
2. **m3_labels** — `data/m3_labels/*.npy` + `manifest.jsonl` (from `labels.py`, ~650 MB).
3. **features** — the BioViL-T cache `<image_id>.npy` (from `extract_features.py`, ~49 GB).

Settings → Accelerator **GPU**, then Run All.

## CONFIG — edit these

In [ ]:
import os
PHASE3_SRC = "/kaggle/input/phase3-code/phase_3"
LABELS     = "/kaggle/input/m3-labels"            # folder with region_concepts.npy ... manifest.jsonl
FEATURES   = "/kaggle/input/mimic-biovilt-feats"  # folder of <image_id>.npy

MODE     = "B"        # C (direct) | A (pure concept bottleneck) | B (hybrid)
EPOCHS   = 40
RUN_NAME = "m3_B"
USE_GLOBAL = 0        # 1 = add BioViL-T global embedding as a 30th region
RESUME   = 0          # 1 from the 2nd session on

RCLONE_REMOTE = "dhint:CHEX-DATA/m3_runs"
SYNC_EVERY    = 0     # 0 = push each epoch; >0 = also push every N steps
RCLONE_SECRET = "RCLONE_CONF"

for k, v in dict(PHASE3_SRC=PHASE3_SRC, LABELS=LABELS, FEATURES=FEATURES, MODE=MODE,
                 EPOCHS=EPOCHS, RUN_NAME=RUN_NAME, USE_GLOBAL=USE_GLOBAL, RESUME=RESUME,
                 RCLONE_REMOTE=RCLONE_REMOTE, SYNC_EVERY=SYNC_EVERY, RCLONE_SECRET=RCLONE_SECRET).items():
    os.environ[k] = str(v)
print("config set | mode", MODE, "| resume", RESUME)

## 1 — rclone + Drive config (graceful: skips sync if no secret)
One-time: copy your local `rclone.conf` text into a Kaggle Secret named `RCLONE_CONF`.

In [ ]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working && curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

In [ ]:
import os, pathlib
SYNC_OK = "0"
try:
    from kaggle_secrets import UserSecretsClient
    conf = UserSecretsClient().get_secret(os.environ["RCLONE_SECRET"])
    cfg = pathlib.Path.home() / ".config" / "rclone"; cfg.mkdir(parents=True, exist_ok=True)
    (cfg / "rclone.conf").write_text(conf)
    if os.system('rclone mkdir "%s"' % os.environ["RCLONE_REMOTE"]) == 0:
        SYNC_OK = "1"; print("remote OK ->", os.environ["RCLONE_REMOTE"])
    else:
        print("[WARN] remote unreachable -> training WITHOUT Drive sync")
except Exception as e:
    print("[WARN] no RCLONE_CONF secret -> training WITHOUT Drive sync:", e)
os.environ["SYNC_OK"] = SYNC_OK; print("SYNC_OK =", SYNC_OK)

## 2 — copy code in

In [ ]:
!rm -rf /kaggle/working/phase_3 && cp -r "$PHASE3_SRC" /kaggle/working/phase_3
%cd /kaggle/working/phase_3
!python -c "import constants as C; print('regions',C.NUM_REGIONS,'concepts',C.NUM_CONCEPTS)"  # sanity

## 3 — train (Drive-synced)

Re-run with `RESUME=1` in CONFIG after a session dies — it pulls `last.pt` from Drive
and continues. Run modes C, A, B (change `MODE`/`RUN_NAME`) to compare accuracy vs explainability.

In [ ]:
import os
G = "--use-global" if int(os.environ["USE_GLOBAL"]) else ""
R = "--resume" if int(os.environ["RESUME"]) else ""
S = ('--sync-remote "%s" --sync-every %s' % (os.environ["RCLONE_REMOTE"], os.environ["SYNC_EVERY"]))     if os.environ.get("SYNC_OK") == "1" else ""
os.environ.update(G_FLAG=G, R_FLAG=R, S_FLAG=S)
print("flags:", G, R, S or "(no sync)")
!python train.py --mode "$MODE" --labels-dir "$LABELS" --features-root "$FEATURES" \
  --out /kaggle/working/m3_runs --name "$RUN_NAME" --epochs $EPOCHS --device cuda \
  --workers 4 $G_FLAG $S_FLAG $R_FLAG

## 4 — eval (test = gold) + infer (per-region JSON for M4/M5)

In [ ]:
!python eval.py --ckpt /kaggle/working/m3_runs/$RUN_NAME/best.pt \
  --labels-dir "$LABELS" --features-root "$FEATURES" --split test --device cuda

In [ ]:
!python infer.py --ckpt /kaggle/working/m3_runs/$RUN_NAME/best.pt \
  --labels-dir "$LABELS" --features-root "$FEATURES" --split test \
  --out /kaggle/working/m3_pred_$RUN_NAME.jsonl --device cuda
import os
if os.environ.get("SYNC_OK")=="1":
    os.system('rclone copy /kaggle/working/m3_runs/%s "%s/%s" --quiet' % (os.environ["RUN_NAME"], os.environ["RCLONE_REMOTE"], os.environ["RUN_NAME"]))
    print("final run pushed to Drive")